In [ ]:
import os
import glob
import numpy as np
import xarray as xr

### Functions

In [ ]:
def calc_event_day(da, threshold, min_days, event_type, default_day=None, doy_start=1):
    """
    Calculate the day of year when a sea ice event occurs per pixel.

    Parameters:
    -----------
    da : xarray.DataArray
        Sea ice concentration [0–1], must have 'time' dimension.
    threshold : float
        Threshold to define ice presence or absence.
    min_days : int
        Minimum consecutive days meeting condition (rolling window size).
    event_type : str
        'ice_free' = first time SIC stays below threshold for min_days.
        'freeze_up' = first time SIC stays above threshold for min_days.
    default_day : int or None
        Value to assign if event never occurs (e.g., perennial ice or always open water).
    doy_start : int
        Starting day of year for your dataset's first time step (e.g. 121 for May 1).

    Returns:
    --------
    event_doy : xarray.DataArray
        Day of year of event occurrence, same spatial dims as input.
    """

    da = da.set_index(time='time')

    # Construct rolling window on time dim
    rolling = da.rolling(time=min_days, center=False).construct("window_dim")

    if event_type == 'ice_free':
        cond = (rolling < threshold).all('window_dim')
    elif event_type == 'freeze_up':
        cond = (rolling > threshold).all('window_dim')
    else:
        raise ValueError("Event_type must be 'ice_free' or 'freeze_up'")

    event_idx = cond.argmax(dim='time')

    # If event never occurs, argmax returns 0 but condition.any is False
    event_idx = event_idx.where(cond.any(dim='time'), other=np.nan)

    # Convert index to day of year with rolling window adjustment
    event_doy = event_idx + doy_start + min_days - 1

    # Mask pixels missing in original data on first time step
    event_doy = event_doy.where(da.isel(time=0).notnull())

    if default_day is not None:
        event_doy = event_doy.fillna(default_day)

    event_doy.name = f"{event_type}_day"
    return event_doy


def calc_ice_free_days(da, threshold):
    """
    Count the number of days where SIC < threshold per pixel.

    Parameters:
    -----------
    da : xarray.DataArray
        Sea ice concentration [0–1], with time dimension.
    threshold : float
        Threshold below which pixel is considered ice free.

    Returns:
    --------
    ice_free_days : xarray.DataArray
        Count of ice free days per pixel.
    """
    ice_free_days = (da < threshold).sum(dim='time')
    ice_free_days.name = 'ice_free_days'
    return ice_free_days


def calc_breakup_day(da, threshold, min_days, doy_start=1, default_day=None):
    """
    Calculate the break-up day: first time after peak winter ice when SIC drops below threshold.

    Parameters:
    -----------
    da : xarray.DataArray
        Sea ice concentration [0–1], must have 'time' dim.
    threshold : float
        SIC threshold for ice break-up.
    min_days : int
        Rolling window size for condition.
    doy_start : int
        DOY of first time step.
    default_day : int or None
        Value to assign if break-up never occurs.

    Returns:
    --------
    breakup_day : xarray.DataArray
        DOY of ice break-up event.
    """
    da = da.set_index(time='time')

    # Rolling window
    rolling = da.rolling(time=min_days, center=False).construct("window_dim")

    cond = (rolling < threshold).all("window_dim")

    event_idx = cond.argmax(dim="time")
    event_idx = event_idx.where(cond.any(dim="time"), other=np.nan)

    breakup_doy = event_idx + doy_start + min_days - 1
    breakup_doy = breakup_doy.where(da.isel(time=0).notnull())

    if default_day is not None:
        breakup_doy = breakup_doy.fillna(default_day)

    breakup_doy.name = "ice_breakup_day"
    return breakup_doy


def process_dataset(sic_da, doy_start=121, freeze_threshold=0.30, freeze_min_days=6,
                    icefree_threshold=0.15, icefree_min_days=1, default_freeze_day=275):
    """
    Compute freeze-up day, ice-free day, and ice-free day count for a dataset.

    Parameters:
    -----------
    sic_da : xarray.DataArray
        Sea ice concentration [0–1] with time dimension.
    doy_start : int
        Day of year corresponding to first time step in sic_da.
    freeze_threshold : float
        Threshold to define freeze-up.
    freeze_min_days : int
        Rolling window size for freeze-up condition.
    icefree_threshold : float
        Threshold to define ice-free.
    icefree_min_days : int
        Rolling window size for ice-free condition.
    default_freeze_day : int
        Day of year assigned if freeze-up never occurs.

    Returns:
    --------
    xr.Dataset
        Dataset containing freeze_up_day, ice_free_day, ice_free_days variables.
    """

    freeze_up_day = calc_event_day(
        sic_da,
        threshold=freeze_threshold,
        min_days=freeze_min_days,
        event_type='freeze_up',
        default_day=default_freeze_day,
        doy_start=doy_start
    )

    ice_free_day = calc_event_day(
        sic_da,
        threshold=icefree_threshold,
        min_days=icefree_min_days,
        event_type='ice_free',
        default_day=None,
        doy_start=doy_start
    )

    ice_free_days = calc_ice_free_days(
        sic_da,
        threshold=icefree_threshold
    )

    return xr.Dataset({
        'freeze_up_day': freeze_up_day,
        'ice_free_day': ice_free_day,
        'ice_free_days': ice_free_days
    })

### Procedural

In [ ]:
# === User config: adjust these variables ===
base_path = '/glade/work/skygale/data/combined/'  # folder with .nc files
output_path = '/glade/work/skygale/output/metrics/'        # where to save results
var_name = 'sea_ice_concentration'                         # SIC variable name in your datasets
doy_start = 121                                            # DOY for first timestep (e.g. May 1)

freeze_threshold = 0.30
freeze_min_days = 6
icefree_threshold = 0.15
icefree_min_days = 1
default_freeze_day = 275

os.makedirs(output_path, exist_ok=True)

# Get list of .nc files
all_files = sorted(glob.glob(os.path.join(base_path, '*.nc')))

for filepath in all_files:
    print(f"Processing {os.path.basename(filepath)} ...")
    try:
        ds = xr.open_dataset(filepath)
        if var_name not in ds:
            print(f"Variable '{var_name}' not found in {filepath}. Skipping.")
            continue

        sic = ds[var_name]

        # Normalize SIC if > 1 (e.g. 0-100%)
        if sic.max() > 1:
            sic = sic / 100.0

        metrics_ds = process_dataset(
            sic_da=sic,
            doy_start=doy_start,
            freeze_threshold=freeze_threshold,
            freeze_min_days=freeze_min_days,
            icefree_threshold=icefree_threshold,
            icefree_min_days=icefree_min_days,
            default_freeze_day=default_freeze_day
        )

        # Add coords for lat/lon if available
        for coord in ['lat', 'latitude', 'lon', 'longitude']:
            if coord in ds.coords:
                metrics_ds = metrics_ds.assign_coords({coord: ds.coords[coord]})

        # Save to netCDF
        fname_out = os.path.basename(filepath).replace('.nc', '_metrics.nc')
        save_path = os.path.join(output_path, fname_out)
        metrics_ds.to_netcdf(save_path)

        print(f"Saved metrics to {save_path}\n")
    except Exception as e:
        print(f"Error processing {filepath}: {e}\n")
